# B06-02 | 실제 OD600 성장곡선과 배지 조건 분석 (Clean v2)

**수정 범위:** v1에서 승인된 계산 정의와 원본 분석 흐름은 유지하고, 품질 플래그를 **생성한 다음 최종 CSV에 저장**합니다. 별도 출력 디렉터리를 사용하며 기존 v1은 덮어쓰지 않습니다.

**검증 원칙:** B06-01 연결 승인이 필수이고 실제 곡선과 배지 평가의 연결은 문자열 키로만 검사합니다. 파일별 SHA256을 기록하고, 디스크에 저장된 CSV를 재조회하여 플래그의 존재와 일관성을 독립적으로 확인합니다. 결측값이나 성장모델의 공식 정의를 추측하지 않습니다.

**이번 단계의 한계:** 결측 208곡선 등은 이전 v1의 실제 실행에서 관측된 값입니다. Clean v2의 검증은 사용자 PC에서 재실행한 뒤 완료됩니다.


## 1. 환경·경로 및 승인 게이트
**작업:** 원본 파일과 B06-01 검증 결과가 갖춰졌는지 확인한다. 검증되지 않은 라벨 매칭으로 데이터를 결합하지 않도록 분석을 중단할 수 있는 게이트이다.

In [ ]:
# 1. 경로 확인: notebooks 폴더/프로젝트 루트에서 실행 가능
from pathlib import Path
import re, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd=Path.cwd().resolve()
candidates=[cwd,cwd.parent,cwd/'bioimage-ai']
ROOT=next((p for p in candidates if (p/'data'/'bioculture'/'media_aida_ying_2025').is_dir()),None)
if ROOT is None:
    raise FileNotFoundError('data/bioculture/media_aida_ying_2025 경로를 찾지 못했습니다.')
DATA=ROOT/'data'/'bioculture'/'media_aida_ying_2025'
AUDIT=ROOT/'results'/'bioculture'/'B06_01_link_audit_v2'
DEST=ROOT/'results'/'bioculture'/'B06_02_growth_curves_v2'
DEST.mkdir(parents=True,exist_ok=True)
ROUND_FILES=sorted(DATA.glob('BW25113_Growth_Round*.xlsx'))
MEDIUM_FILE=DATA/'BW25113_Medium composition.xlsx'
EVAL_FILE=DATA/'BW25113_GrowthDataEvaluation.xlsx'
MAP_FILE=AUDIT/'VALIDATED_curve_assay_condition_mapping.csv'
REPORT_FILE=AUDIT/'link_audit_report_v2.json'
assert len(ROUND_FILES)==7 and all(p.exists() for p in [MEDIUM_FILE,EVAL_FILE,MAP_FILE,REPORT_FILE])
report=json.loads(REPORT_FILE.read_text(encoding='utf-8'))
assert report.get('join_validated') is True and report.get('scientific_analysis_ready') is True, 'B06-01 연결 미승인'
print('Root:',ROOT,'\nOutput:',DEST)
print('Approved mapping:', MAP_FILE)
print('Clean v2: preserve existing B06_02_growth_curves (v1) outputs')

## 2. 검증된 매핑과 원자료 메타데이터 확인
**작업:** 매핑의 13,608개 키가 고유하며 배지·성장 평가표와 일치하는지 다시 확인한다. 이전 단계의 상태 플래그만 신뢰하지 않고 이번 분석에서도 원본 메타데이터를 재검증한다.

In [ ]:
# 2. 검증된 키 매핑 로드, 메타데이터 재결합
mapping=pd.read_csv(MAP_FILE,dtype={'Label':'string','Assay ID':'string','Condition ID':'string'})
medium=pd.read_excel(MEDIUM_FILE,engine='openpyxl',dtype={'Label':'string','Assay ID':'string','Condition ID':'string'})
evaluation=pd.read_excel(EVAL_FILE,engine='openpyxl',dtype={'Label':'string','Assay ID':'string','Condition ID':'string'})
keys=['Label','Assay ID','Condition ID']
assert len(mapping)==13608 and mapping.duplicated(['round','Label']).sum()==0
assert medium.duplicated(keys).sum()==0 and evaluation.duplicated(keys).sum()==0
assert all(k in mapping for k in ['round']+keys)
assert not mapping[keys+['round']].isna().any().any()
# verify each mapping row exists in BOTH source metadata tables
mapped_meta=mapping.merge(medium[keys],on=keys,how='left',indicator=True,validate='one_to_one')
assert mapped_meta['_merge'].eq('both').all()
mapped_eval=mapping.merge(evaluation[keys],on=keys,how='left',indicator=True,validate='one_to_one')
assert mapped_eval['_merge'].eq('both').all()
print('Verified mapping rows:',len(mapping),'Unique conditions:',mapping['Condition ID'].nunique())
print('Evaluation fields:',list(evaluation.columns))
display(mapping.head())

## 3. 곡선 품질·기초 지표 계산 함수
**작업:** 결측 관측점 수, 첫/마지막 유효 시점, 값 범위, 최대 관측값, 기초 구간과 말기 구간의 평균값, 단순 변화량을 계산한다. 지표는 **운영상 정의한 기술통계**이며 `K`, `r`, 생물학적 수용력 또는 세대시간으로 간주하지 않는다.

`growth_log_slope_per_h`는 양수 관측값의 로그를 회귀한 **탐색적 전체 구간 기울기**로, 지수성장 구간을 선별한 공식 최대 성장률이 아니다.

In [ ]:
# 3. 개별 곡선에 대한 NaN-aware 기초 요약

def summarize_curve(t, y, label, round_id):
    valid=np.isfinite(t)&np.isfinite(y)
    tv=t[valid]; yv=y[valid]
    row={'round':int(round_id),'Label':str(label),
         'n_total_timepoints':len(t),'n_valid':int(valid.sum()),
         'missing_fraction':float(1-valid.mean()),
         'time_first_h':np.nan,'time_last_h':np.nan,
         'first_value':np.nan,'last_value':np.nan,
         'observed_min':np.nan,'observed_max':np.nan,
         'delta_last_first':np.nan,'early_mean':np.nan,'late_mean':np.nan,
         'growth_log_slope_per_h':np.nan,'log_fit_n':0}
    if len(yv)==0:return row
    row.update(time_first_h=float(tv[0]),time_last_h=float(tv[-1]),
        first_value=float(yv[0]),last_value=float(yv[-1]),
        observed_min=float(yv.min()),observed_max=float(yv.max()),
        delta_last_first=float(yv[-1]-yv[0]))
    # 유효 관측값 앞/뒤 최대 3개 평균: 규약적 지표, 실험 구간의 이름은 아님
    k=min(3,len(yv))
    row['early_mean']=float(np.mean(yv[:k]))
    row['late_mean']=float(np.mean(yv[-k:]))
    positive=(yv>0)
    if positive.sum()>=3 and np.ptp(tv[positive])>0:
        row['growth_log_slope_per_h']=float(np.polyfit(tv[positive],np.log(yv[positive]),1)[0])
        row['log_fit_n']=int(positive.sum())
    return row

# 코드 자체 단위 테스트: 결측이 있어도 유효 관측만 계산하는가?
_demo=summarize_curve(np.array([0.,1.,2.,3.]),np.array([.1,np.nan,.2,.4]),'demo',1)
assert _demo['n_valid']==3 and abs(_demo['observed_max']-.4)<1e-12
print('Function self-check passed')

## 4. 전체 7개 라운드의 원시 곡선 요약
**작업:** Excel을 한 파일씩 읽어 약 13,608개 곡선을 요약한다. 전체 자료를 130만 행 이상의 Long format으로 펼치지 않아 메모리 부담을 줄인다. **관측값 결측을 0으로 채우거나 선형 보간하지 않는다.**

In [ ]:
# 4. 라운드별 요약: 시간축 재검사 / 전역 Label 확인
rows=[]; round_qc=[]
for f in ROUND_FILES:
    rid=int(re.search(r'Round(\d+)',f.name,re.I).group(1))
    data=pd.read_excel(f,engine='openpyxl')
    timecol=next((c for c in data if str(c).strip().lower()=='time (h)'),None)
    assert timecol is not None,f'{f.name} 시간 열 누락'
    t=pd.to_numeric(data[timecol],errors='coerce').to_numpy(dtype=float)
    assert np.isfinite(t).all() and (np.diff(t)>0).all(),f'{f.name} 시간축 이상'
    labels=[c for c in data if re.fullmatch(r'Curve\d+',str(c),re.I)]
    expected=set(mapping.loc[mapping['round']==rid,'Label'].astype(str))
    assert set(map(str,labels))==expected and len(labels)==len(expected),f'Round{rid} Label 대응 실패'
    numeric=data[labels].apply(pd.to_numeric,errors='coerce')
    for label in labels:
        rows.append(summarize_curve(t,numeric[label].to_numpy(dtype=float),label,rid))
    round_qc.append({'round':rid,'n_curves':len(labels),'n_timepoints':len(t),
                     'time_min_h':float(t.min()),'time_max_h':float(t.max()),
                     'missing_fraction':float(numeric.isna().mean().mean())})
    print(f.name, 'curves',len(labels),'NaN fraction',round_qc[-1]['missing_fraction'])
curves=pd.DataFrame(rows)
round_qc=pd.DataFrame(round_qc)
assert len(curves)==len(mapping)==13608 and not curves.duplicated(['round','Label']).any()
display(round_qc)
print('Valid points per curve:',curves['n_valid'].describe().to_dict())
curves.to_csv(DEST/'raw_curve_summary.csv',index=False,encoding='utf-8-sig')
round_qc.to_csv(DEST/'round_curve_quality.csv',index=False,encoding='utf-8-sig')

## 5. 승인된 키로 배지 조건·원래 성장 평가 값 연결
**작업:** `validate='one_to_one'`를 이용해 잘못된 다중 결합을 방지하고, 누락 없이 연결되었는지 검사한다. 배지 조성 47열 전체는 별도 원본에 유지하고, 이 Notebook의 곡선별 CSV에는 식별자·기초 지표·기존 K/r 및 정보 코드를 기록한다.

In [ ]:
# 5. 데이터 연결: 반드시 B06-01 승인 매핑을 경유
joined=curves.merge(mapping[['round']+keys],on=['round','Label'],how='left',validate='one_to_one',indicator=True)
assert joined['_merge'].eq('both').all()
joined=joined.drop(columns='_merge')
eval_cols=[c for c in ['K','r','K_info','r_info'] if c in evaluation.columns]
joined=joined.merge(evaluation[keys+eval_cols],on=keys,how='left',validate='one_to_one',indicator=True)
assert joined['_merge'].eq('both').all() and len(joined)==len(curves)
joined=joined.drop(columns='_merge')
assert joined['Condition ID'].nunique()==mapping['Condition ID'].nunique()
print('Joined curves:',len(joined),'conditions:',joined['Condition ID'].nunique())
display(joined[['round']+keys+['n_valid','observed_max','delta_last_first']+eval_cols].head(8))
# 최종 결과 CSV는 QC 플래그를 모두 추가한 후에만 저장한다 (셀 6).

## 6. 관측 품질과 라운드 차이
**작업:** 빈 곡선, 관측점 부족, 음수 관측값, 관측 시간이 짧은 곡선을 표시한다. 이 품질 표시는 **교육용 진단 규칙**이며 원본 연구의 `K_info/r_info` 코드를 대체하지 않는다.
**Clean v2:** 빈 곡선의 관측 시간폭은 판단 불가(`flag_window_unknown`)로 구분하고 최종 연결 CSV를 이 셀에서 저장한다. `flag_model_candidate`는 후속 모델의 **탐색적 입력 후보**일 뿐 모델 적합 성공을 뜻하지 않는다.


In [ ]:
# 6. 품질 플래그와 라운드/조건별 분포
MIN_VALID=8  # 민감도 분석 대상으로도 활용 가능; 공식 실험 QC 기준은 아님
joined['flag_empty']=joined['n_valid']==0
joined['flag_sparse']=joined['n_valid']<MIN_VALID
joined['flag_negative_observed']=joined['observed_min']<0
# 빈 곡선은 관측 시간폭 판단이 불가능하므로 별도 상태로 보존한다.
joined['observed_window_h']=joined['time_last_h']-joined['time_first_h']
joined['flag_window_unknown']=joined['observed_window_h'].isna()
joined['flag_short_window']=joined['observed_window_h'].lt(12).fillna(False).astype(bool)
joined['flag_model_candidate']= (~joined['flag_sparse']) & (~joined['flag_window_unknown']) & (~joined['flag_short_window'])
assert (joined.loc[joined['flag_empty'],'flag_window_unknown']).all()
assert (~joined.loc[joined['flag_empty'],'flag_short_window']).all()
assert (joined['flag_model_candidate'] & joined['flag_sparse']).sum()==0
qc_by_round=joined.groupby('round',as_index=False).agg(
 curves=('Label','size'),empty=('flag_empty','sum'),sparse=('flag_sparse','sum'),
 negative=('flag_negative_observed','sum'),short_window=('flag_short_window','sum'),
 median_valid=('n_valid','median'),median_observed_peak=('observed_max','median'))
qc_by_round['unknown_window']=joined.groupby('round')['flag_window_unknown'].sum().to_numpy()
qc_by_round['model_candidates']=joined.groupby('round')['flag_model_candidate'].sum().to_numpy()
display(qc_by_round)
# v1의 저장 순서 문제 수정: 반드시 모든 품질 컬럼 추가 후 전체 CSV 기록.
joined.to_csv(DEST/'VALIDATED_curve_metrics_with_metadata.csv',index=False,encoding='utf-8-sig')
qc_by_round.to_csv(DEST/'curve_qc_by_round.csv',index=False,encoding='utf-8-sig')
print('Saved authoritative joined CSV with QC flags:',DEST/'VALIDATED_curve_metrics_with_metadata.csv')

## 7. 배지 조건별 반복 관측 요약
**작업:** `Condition ID`별로 배지 조성은 동일하더라도 측정 라운드와 관찰 품질이 다를 수 있음을 점검한다. 반복 관측값은 생물학적 독립 반복과 자동으로 동일시하지 않는다.

In [ ]:
# 7. 조건별 반복 수와 기초 관측 지표 분포
condition_summary=joined.groupby('Condition ID',as_index=False).agg(
    n_curves=('Label','size'),n_rounds=('round','nunique'),
    n_usable=('flag_sparse',lambda s:int((~s).sum())),
    median_peak_observed=('observed_max','median'),
    median_change_observed=('delta_last_first','median'),
    median_valid_points=('n_valid','median'))
assert len(condition_summary)==1029
print('Repetitions per condition:',condition_summary['n_curves'].describe().to_dict())
display(condition_summary.sort_values('n_curves',ascending=False).head(10))
condition_summary.to_csv(DEST/'condition_replicate_summary.csv',index=False,encoding='utf-8-sig')

## 8. 원시 성장곡선 대표 사례 시각화
**작업:** 유효 관측점이 충분한 곡선 몇 개를 선택해 실제 시간–측정값을 표시한다. **보간선 대신 실제 점만 표시**하며, 비슷한 조건에 속한 곡선도 곡선 모양이 달라질 수 있음을 관찰한다.

In [ ]:
# 8. 재현 가능한 선택 규칙: 유효 관측값 많은 곡선을 우선, 라운드 다양성 확보
selected=(joined.loc[joined['n_valid']>=20].sort_values(['round','n_valid','Label'],ascending=[True,False,True])
          .groupby('round',as_index=False).head(1).head(7))
fig,ax=plt.subplots(figsize=(9,5))
for rid,group in selected.groupby('round'):
    f=next(f for f in ROUND_FILES if int(re.search(r'Round(\d+)',f.name,re.I).group(1))==rid)
    df=pd.read_excel(f,engine='openpyxl')
    label=str(group.iloc[0]['Label'])
    time=pd.to_numeric(df['Time (h)'],errors='coerce')
    y=pd.to_numeric(df[label],errors='coerce')
    m=time.notna() & y.notna()
    ax.plot(time[m],y[m],'.-',markersize=3,label=f'R{rid}: {label}')
ax.set(xlabel='Time (h)',ylabel='Recorded growth-curve value',title='Observed growth curves (selected examples)')
ax.legend(fontsize=7,ncol=2);ax.grid(alpha=.2)
plt.tight_layout();fig.savefig(DEST/'example_raw_growth_curves.png',dpi=160);plt.show()

## 9. 기존 K/r과 기초 지표의 탐색적 비교
**작업:** 기존 `K`와 실제 관측 최대값, 기존 `r`과 기초 로그 기울기의 산점도를 만든다. 이 상관은 동일한 성장 지표라는 증거가 아니며, 기존 논문의 `K/r` 정의를 대신하지 않는다.

In [ ]:
# 9. 용어 정의가 검증되기 전에는 '상관'을 기술통계로만 보고
fig,axes=plt.subplots(1,2,figsize=(11,4))
comparisons=[('K','observed_max','K vs observed maximum'),
             ('r','growth_log_slope_per_h','r vs exploratory log slope')]
comparison_stats=[]
for ax,(x,y,title) in zip(axes,comparisons):
    tmp=joined[[x,y]].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
    if len(tmp)>0:
        ax.scatter(tmp[x],tmp[y],s=5,alpha=.16)
    rho=float(tmp[x].corr(tmp[y])) if len(tmp)>2 else np.nan
    comparison_stats.append({'reference_field':x,'empirical_field':y,'paired_n':len(tmp),'pearson_r':rho})
    ax.set(xlabel=f'Source {x} (definition not yet verified)',ylabel=y,title=title)
    ax.grid(alpha=.2)
plt.tight_layout();fig.savefig(DEST/'reference_metrics_exploratory.png',dpi=160);plt.show()
comparison_stats=pd.DataFrame(comparison_stats)
display(comparison_stats)
comparison_stats.to_csv(DEST/'reference_metric_comparison.csv',index=False,encoding='utf-8-sig')

## 10. 검증 보고서 및 다음 연구 단계
**작업:** 분석에 쓰인 연결 승인 정보와 데이터 특징을 기계적으로 기록한다. 결과는 **B06-02의 표준화된 기초 감사 산출물**이며, B06 후속 Notebook에서 지연시간·최대 성장률을 추정하기 전에 품질 기준을 결정하는 데 사용한다.
**Clean v2:** 저장된 결과를 재조회해 QC 플래그의 존재와 값 일치를 확인하고 입력·출력 SHA256 명세서를 생성한다.


In [ ]:
# 10. 입력 데이터·연결키 및 결과 일관성 최종 검사
assert len(joined)==13608 and len(condition_summary)==1029
assert joined[['round','Label']].duplicated().sum()==0
assert joined[keys].isna().sum().sum()==0
assert (joined['n_valid']<=joined['n_total_timepoints']).all()
# 독립 디스크 재조회: 세션 내 DataFrame의 존재만으로 저장 성공이라 판단하지 않는다.
saved=pd.read_csv(DEST/'VALIDATED_curve_metrics_with_metadata.csv',encoding='utf-8-sig',low_memory=False)
required_flags=['flag_empty','flag_sparse','flag_negative_observed','flag_short_window',
                'flag_window_unknown','flag_model_candidate']
assert len(saved)==len(joined) and all(k in saved.columns for k in required_flags)
assert saved[['round','Label']].duplicated().sum()==0
assert saved[['round','Label']].astype(str).equals(joined[['round','Label']].astype(str).reset_index(drop=True))
for flag in required_flags:
    # 판다스 CSV bool 타입 추론을 문자열에 의존하지 않고 정규화
    disk=saved[flag].astype(str).str.lower().map({'true':True,'false':False})
    assert disk.notna().all(),f'{flag}: 예상 밖의 저장 값'
    assert (disk.to_numpy()==joined[flag].to_numpy()).all(),f'{flag}: 저장 값 불일치'
assert int(joined['flag_empty'].sum())==int((joined['n_valid']==0).sum())
assert int(joined['flag_sparse'].sum())==int((joined['n_valid']<MIN_VALID).sum())
assert int(joined['flag_window_unknown'].sum())==int(joined['observed_window_h'].isna().sum())
print('Disk round-trip and QC flag checks passed')
expected_outputs=['raw_curve_summary.csv','round_curve_quality.csv',
 'VALIDATED_curve_metrics_with_metadata.csv','curve_qc_by_round.csv',
 'condition_replicate_summary.csv','example_raw_growth_curves.png',
 'reference_metrics_exploratory.png','reference_metric_comparison.csv']
assert all((DEST/name).exists() for name in expected_outputs)
summary={
 'data_set':'Aida–Ying BW25113',
 'mapping_source':str(MAP_FILE),'mapping_approved':True,
 'n_curves':int(len(joined)), 'n_conditions':int(len(condition_summary)),
 'n_rounds':int(joined['round'].nunique()),
 'empty_curves':int(joined['flag_empty'].sum()),
 'sparse_curves_lt_8_valid_points':int(joined['flag_sparse'].sum()),
 'median_valid_points':float(joined['n_valid'].median()),
 'qc_min_valid_threshold':MIN_VALID,
 'short_window_threshold_h':12,
 'short_window_unknown_curves':int(joined['flag_window_unknown'].sum()),
 'model_candidate_curves_exploratory':int(joined['flag_model_candidate'].sum()),
 'qc_flags_persisted_and_verified':True,
 'authoritative_metrics_csv':'VALIDATED_curve_metrics_with_metadata.csv',
 'source_metric_definitions_verified':False,
 'caveat':'기초 관측 기술통계 및 탐색적 비교. K/r 정의, 반복 독립성, OD 보정 및 생물학적 해석은 추가 검증 필요.'}
(DEST/'B06_02_validation_report.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('Saved to:',DEST)
# 산출물 및 입력 원본 추적: 해시는 입력 바이트 동일성만 증명하며 생물학적 정확성은 보장하지 않음.
input_paths=ROUND_FILES+[MEDIUM_FILE,EVAL_FILE,MAP_FILE,REPORT_FILE]
def sha256_file(path,blocksize=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as file:
        for chunk in iter(lambda:file.read(blocksize),b''):h.update(chunk)
    return h.hexdigest()
input_manifest=pd.DataFrame([{'file':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in input_paths])
input_manifest.to_csv(DEST/'input_manifest_sha256.csv',index=False,encoding='utf-8-sig')
output_manifest=pd.DataFrame([{'file':p.name,'size_bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in sorted(DEST.iterdir()) if p.is_file() and p.name not in ('output_manifest_sha256.csv',)])
output_manifest.to_csv(DEST/'output_manifest_sha256.csv',index=False,encoding='utf-8-sig')
assert (DEST/'input_manifest_sha256.csv').exists()
assert (DEST/'output_manifest_sha256.csv').exists()
print('Provenance manifests written:',len(input_manifest),'inputs,',len(output_manifest),'outputs')


## 연습문제 — 정답은 별도 Word 실습 해설서에 수록 예정
1. `MIN_VALID`를 8에서 12로 변경하면 라운드별 `sparse` 수가 어떻게 달라지는가? 이 차이가 무엇을 의미하는가?
2. Round별 결측 비율이 크게 다른 이유를 데이터 구조 측면에서 어떻게 조사할 수 있는가?
3. 같은 `Condition ID`의 반복 곡선을 두 개 그려 형태 차이를 해석하라. 생물학적 반복이라고 단정할 수 있는가?
4. `observed_max`와 원본 `K`가 일치하지 않을 수 있는 이유를 설명하라.
5. 기록된 OD 값이 0 이하일 때 로그 변환을 함부로 적용해서는 안 되는 이유를 설명하라.

**다음:** B06-02 실제 결과 QC → 후속 B06-03의 성장 특성 추출(지연시간·최대 성장률 등)을 위한 원논문 정의 확인 및 알고리즘 검증.